# 面试问题：MCP 的 Host、Client、Server、Tools、Resources、Prompts 和生命周期是什么，怎样实现一个最小协议子集？

**一句话回答**：Host 管理用户体验、模型与安全策略；每个 Server 对应 Client 会话。MCP 数据层基于 JSON-RPC，先 initialize 协商 protocol version/capabilities，再 initialized；Server 可暴露 tools、resources、prompts，Client 仅调用已协商能力。协议发现不等于授权，Host 仍需用户同意、schema、ACL、取消与不可信结果隔离。

本 Notebook 用标准库实现教学版 JSON-RPC 校验、初始化、分页发现、工具调用、资源读取、progress/cancel 和安全命名空间，不冒充完整 SDK。重点不是背协议名词，而是解释状态机、能力协商、调用边界以及失败后如何安全收敛。

In [ ]:
from dataclasses import dataclass
import hashlib,json,math

SEED125=12501; PROTOCOL125="2025-11-25"
assert SEED125==12501
assert len(PROTOCOL125.split("-"))==3
assert hashlib.sha256(b"client").hexdigest()!=hashlib.sha256(b"server").hexdigest()

## 1. JSON-RPC 区分 request、response 和 notification

request 有 `jsonrpc=2.0/id/method/params`，response 恰有 result 或 error；notification 无 id、无需 response。ID 在未决请求中唯一，未知字段和畸形消息拒绝。Transport 可为 stdio 或 Streamable HTTP，但数据语义相同。

In [ ]:
def validate_request125(msg):
    if not isinstance(msg,dict) or msg.get("jsonrpc")!="2.0" or not isinstance(msg.get("method"),str): return False
    allowed={"jsonrpc","id","method","params"}
    return not (set(msg)-allowed) and ("params" not in msg or isinstance(msg["params"],dict))
req125={"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":PROTOCOL125}}
note125={"jsonrpc":"2.0","method":"notifications/initialized","params":{}}
assert validate_request125(req125) and validate_request125(note125)
assert "id" not in note125
assert not validate_request125({"jsonrpc":"1.0","id":1,"method":"x"})

## 2. Initialize 协商版本与 capability

Client 声明支持的版本和 client capabilities，Server 选择兼容版本并返回 server capabilities/info；随后 Client 发 initialized notification。调用未声明能力是协议错误。版本不兼容应失败，不可默默降级到未知语义。

In [ ]:
SERVER125={"versions":{"2025-06-18","2025-11-25"},"capabilities":{"tools":{"listChanged":True},"resources":{"subscribe":False},"prompts":{}},"serverInfo":{"name":"demo","version":"1.0"}}
def initialize125(client_versions,client_caps):
    common=sorted(set(client_versions)&SERVER125["versions"],reverse=True)
    if not common: return {"error":"unsupported_version"}
    return {"protocolVersion":common[0],"capabilities":SERVER125["capabilities"],"serverInfo":SERVER125["serverInfo"]}
init125=initialize125(["2025-11-25","2024-11-05"],{"sampling":{}})
assert init125["protocolVersion"]=="2025-11-25"
assert set(init125["capabilities"])=={"tools","resources","prompts"}
assert initialize125(["1900-01-01"],{})=={"error":"unsupported_version"}

## 3. Tools、Resources、Prompts 的控制语义不同

Tools 是模型可提议调用的动作，Resources 是应用附加/读取的上下文，Prompts 是用户选择的模板。都需来源和描述，但不要把 resource 文本当系统指令，也不要把 prompt 当权限。客户端只启用业务需要的 primitive。

In [ ]:
primitives125={"tools":{"control":"model","methods":{"tools/list","tools/call"}},"resources":{"control":"application","methods":{"resources/list","resources/read"}},"prompts":{"control":"user","methods":{"prompts/list","prompts/get"}}}
assert primitives125["tools"]["control"]=="model"
assert primitives125["resources"]["control"]=="application"
assert primitives125["prompts"]["control"]=="user" and "prompts/get" in primitives125["prompts"]["methods"]

## 4. Discovery 使用分页和稳定命名空间

`tools/list` 返回工具定义与 nextCursor；客户端遍历分页、验证唯一名称和 inputSchema，并缓存到 `(server identity, protocol, catalog version)`。多个 Server 同名工具在 Host 内使用 server-qualified ID，避免混淆。

In [ ]:
TOOLS125=[{"name":"search","description":"搜索文档","inputSchema":{"type":"object","required":["query"],"properties":{"query":{"type":"string"}},"additionalProperties":False}},{"name":"calculate","description":"计算表达式","inputSchema":{"type":"object","required":["expression"],"properties":{"expression":{"type":"string"}},"additionalProperties":False}},{"name":"weather","description":"天气","inputSchema":{"type":"object","required":["city"],"properties":{"city":{"type":"string"}},"additionalProperties":False}}]
def tools_list125(cursor=None,page_size=2):
    start=int(cursor or 0); page=TOOLS125[start:start+page_size]; nxt=str(start+page_size) if start+page_size<len(TOOLS125) else None; return {"tools":page,"nextCursor":nxt}
page1_125=tools_list125(); page2_125=tools_list125(page1_125["nextCursor"])
assert [x["name"] for x in page1_125["tools"]]==["search","calculate"]
assert [x["name"] for x in page2_125["tools"]]==["weather"] and page2_125["nextCursor"] is None
assert len({x["name"] for x in TOOLS125})==3

## 5. Tool call 先过 schema、授权和审批

MCP 的 `tools/call` 是协议消息，不是安全许可。Host 根据真实用户/租户过滤工具，校验参数与风险，再决定是否提示审批。Server 返回 structuredContent/content 和 isError；错误结果仍作为数据交给 Agent，不拼成高优先级指令。

In [ ]:
def validate_tool_args125(name,args):
    tool=next((t for t in TOOLS125 if t["name"]==name),None)
    if not tool or not isinstance(args,dict): return False,"unknown_or_shape"
    schema=tool["inputSchema"]; required=set(schema["required"]); props=set(schema["properties"])
    if not required<=set(args) or set(args)-props: return False,"schema"
    if any(not isinstance(v,str) for v in args.values()): return False,"type"
    return True,"valid"
def tool_call125(name,args,scopes):
    ok,reason=validate_tool_args125(name,args)
    if not ok: return {"isError":True,"content":[{"type":"text","text":reason}]}
    if name not in scopes: return {"isError":True,"content":[{"type":"text","text":"forbidden"}]}
    return {"isError":False,"structuredContent":{"tool":name,"accepted":True},"content":[{"type":"text","text":json.dumps({"accepted":True})}]}
assert validate_tool_args125("search",{"query":"MCP"})==(True,"valid")
assert tool_call125("search",{"query":"MCP"},{"search"})["isError"] is False
assert tool_call125("search",{"query":"MCP","extra":"x"},{"search"})["isError"] is True

## 6. Resource URI 经过 Host ACL，内容保持 provenance

Server 列出 resource URI、mime、size 等；读取前 Host 判断用户是否允许把该资源交给模型。`file://`、数据库或远程 URL 都不能绕过宿主权限。返回文本/二进制是 untrusted content，并有大小与媒体类型上限。

In [ ]:
RESOURCES125={"kb://T1/policy":{"tenant":"T1","mimeType":"text/plain","text":"退款7天"},"kb://T2/secret":{"tenant":"T2","mimeType":"text/plain","text":"秘密"}}
def resource_read125(uri,tenant,max_chars=100):
    r=RESOURCES125.get(uri)
    if not r or r["tenant"]!=tenant: return {"error":"forbidden_or_missing"}
    if len(r["text"])>max_chars: return {"error":"too_large"}
    return {"contents":[{"uri":uri,"mimeType":r["mimeType"],"text":r["text"],"untrusted":True}]}
good_resource125=resource_read125("kb://T1/policy","T1")
assert good_resource125["contents"][0]["text"]=="退款7天"
assert good_resource125["contents"][0]["untrusted"]
assert resource_read125("kb://T2/secret","T1")=={"error":"forbidden_or_missing"}

## 7. Long-running call 支持 progress 与 cancellation

progress token 关联原请求，notification 单调更新；取消请求后 Server 尽力停止，但已完成副作用需幂等/补偿。客户端 deadline 到达也发 cancel，并把最终状态记录为 cancelled/timeout，而不是无限等待。

In [ ]:
jobs125={"req-9":{"progress":0,"cancelled":False}}
def progress125(req,value):
    job=jobs125.get(req)
    if not job or job["cancelled"] or not job["progress"]<=value<=100: return False
    job["progress"]=value; return True
def cancel125(req):
    if req not in jobs125: return False
    jobs125[req]["cancelled"]=True; return True
assert progress125("req-9",30) and progress125("req-9",80)
assert not progress125("req-9",50)
assert cancel125("req-9") and not progress125("req-9",90)

## 8. Server 身份、工具变化、同意与供应链治理

第三方 Server 可投毒描述、更新工具或请求递归 sampling。Host 固定 server identity/版本，工具列表变更重新审查；OAuth scope 与业务权限最小化；用户可见并撤销连接。日志避免默认保存 resource/Prompt 敏感正文。

In [ ]:
server_allow125={"server_id":"corp-kb","publisher":"corp","digest":"abc123","allowed_capabilities":{"tools","resources"}}
def capability_allowed125(server,cap): return cap in server["allowed_capabilities"]
manifest125={"schema":1,"protocol":PROTOCOL125,"transport":"stdio-demo","server_identity":"corp-kb","capabilities":["tools","resources"],"tool_ids":"server_qualified","authorization":"host_enforced","content":"untrusted"}; digest125=hashlib.sha256(json.dumps(manifest125,sort_keys=True).encode()).hexdigest()
assert capability_allowed125(server_allow125,"tools")
assert not capability_allowed125(server_allow125,"sampling")
assert len(digest125)==64 and manifest125["authorization"]=="host_enforced"

## 面试总结

回答主线是：**Host/Client/Server → JSON-RPC request/notification → initialize/initialized 能力协商 → tools/resources/prompts 控制语义 → 分页发现 → schema+Host 授权调用 → resource ACL → progress/cancel → Server 身份与用户同意**。MCP 标准化连接协议，不替应用自动解决权限与信任。

延伸阅读：[MCP Architecture](https://modelcontextprotocol.io/docs/learn/architecture)、[MCP Base Protocol](https://modelcontextprotocol.io/specification/2025-06-18/basic/index)、[MCP Tools](https://modelcontextprotocol.io/specification/2025-11-25/server/tools)。